In [1]:
# Sequential

In [2]:
from typing import Dict, Any, List, Union, Optional, Type, Callable, Tuple, Literal, Set
from pydantic import BaseModel, Field
import uuid
from functools import partial

from langgraph.graph import StateGraph, END
from langgraph.types import Command, Send
from langgraph.checkpoint.memory import MemorySaver

from src.haive.core.engine.agent.agent import AgentArchitectureConfig, AgentArchitecture, register_agent
from src.haive.core.engine.aug_llm import AugLLMConfig, compose_runnable
from src.haive.core.models.llm.base import AzureLLMConfig
from src.haive.core.utils.visualize_graph_utils import render_and_display_graph




class SequentialAgentConfig(BaseAgentConfig):
    """
    Configuration for a sequential multi-node agent that extends the BaseAgent.
    
    This agent chains multiple AugLLM configs together in sequence, with optional
    input/output mappings between them.
    """
    nodes: Union[List[Union[AugLLMConfig, Tuple[AugLLMConfig, Dict[str, str], Dict[str, str]]]], 
                Dict[str, Union[AugLLMConfig, Tuple[AugLLMConfig, Dict[str, str], Dict[str, str]]]]] = Field(
        default_factory=list,
        description="List or Dict of AugLLM configs or (config, input_mapping, output_mapping) tuples"
    )
    node_names: Optional[List[str]] = Field(
        default=None,
        description="Optional node names to use (only if nodes is a list)"
    )
    edge_map: Optional[List[Tuple[str, Union[str, Literal["END"]]]]] = Field(
        default=None,
        description="Optional edge map to define routing (required if nodes is a dict)"
    )
    custom_state: Optional[Dict[str, Any]] = Field(
        default=None,
        description="Optional custom state fields to include in the schema"
    )
    
    def build_agent(self) -> "SequentialAgent":
        """Build a SequentialAgent from this configuration."""
        return SequentialAgent(config=self)


#@register_agent(SequentialAgentConfig, None)  # Will be registered at end of file
class SequentialAgent(BaseAgent):
    """
    An agent that chains multiple components in sequence with custom input/output mappings.
    Extends BaseAgent to inherit core agent functionality.
    """
    def __init__(self, config: SequentialAgentConfig):
        # Store configuration specific to SequentialAgent
        self.node_configs = config.nodes
        self.node_names = config.node_names
        self.edge_map = config.edge_map
        self.custom_state = config.custom_state or {"messages": []}
        
        # Initialize base agent
        super().__init__(config)
        
    def setup_workflow(self):
        """
        Set up a workflow based on the chain configuration.
        Override's BaseAgent's simple workflow.
        """
        #from state_schema_manager import StateSchemaManager
        #from node_factory_refined import augment_schema_for_llm, create_node_function
        
        # Start by creating a schema manager from the custom state
        schema_manager = StateSchemaManager(self.custom_state)
        
        # Extract all AugLLM configs
        if isinstance(self.node_configs, dict):
            # Unpack configs from dictionary with tuples
            configs_list = []
            for name, config_item in self.node_configs.items():
                if isinstance(config_item, tuple) and len(config_item) >= 1:
                    configs_list.append(config_item[0])
                else:
                    configs_list.append(config_item)
        else:
            # Unpack configs from list with tuples
            configs_list = []
            for config_item in self.node_configs:
                if isinstance(config_item, tuple) and len(config_item) >= 1:
                    configs_list.append(config_item[0])
                else:
                    configs_list.append(config_item)
        
        # Derive schema from all configs
        for config in configs_list:
            # Augment schema for each config
            schema_manager = augment_schema_for_llm(schema_manager, config)
        
        # Get the schema model
        self.state_schema = schema_manager.get_model()
        self.graph = StateGraph(self.state_schema)
        
        # Handle dictionary of configs with explicit node names
        if isinstance(self.node_configs, dict):
            # Edge map is required for dictionary configs
            if not self.edge_map:
                # Default to sequential if no edge map
                node_names = list(self.node_configs.keys())
                self.edge_map = [(node_names[i], node_names[i+1]) for i in range(len(node_names)-1)]
                # Last node goes to END
                self.edge_map.append((node_names[-1], END))
            
            # First pass: Add all nodes without explicit next_node
            for node_name, config_item in self.node_configs.items():
                # Extract config and mappings
                if isinstance(config_item, tuple) and len(config_item) >= 1:
                    config = config_item[0]
                    input_mapping = config_item[1] if len(config_item) > 1 else None
                    output_mapping = config_item[2] if len(config_item) > 2 else None
                else:
                    config = config_item
                    input_mapping = None
                    output_mapping = None
                
                # Create node function with END as default next node
                node_fn = create_node_function(
                    config=config,
                    input_mapping=input_mapping,
                    output_mapping=output_mapping,
                    next_node=END  # Default to END, will be overridden by edges
                )
                
                # Add node to graph
                self.graph.add_node(node_name, node_fn)
            
            # Second pass: Add all edges to override the default END routing
            for from_node, to_node in self.edge_map:
                if to_node != END:
                    self.graph.add_edge(from_node, to_node)
                else:
                    # Explicitly add edge to END
                    self.graph.add_edge(from_node, END)
                
        # Handle list of configs with auto-generated or explicit names
        else:
            # Generate node names if not provided
            if self.node_names is None:
                self.node_names = [f"node_{i}" for i in range(len(self.node_configs))]
            elif len(self.node_names) < len(self.node_configs):
                # Extend with default names if needed
                self.node_names.extend([f"node_{i+len(self.node_names)}" for i in range(len(self.node_configs) - len(self.node_names))])
            
            # Add nodes with sequential routing
            for i, config_item in enumerate(self.node_configs):
                # Extract config and mappings
                if isinstance(config_item, tuple) and len(config_item) >= 1:
                    config = config_item[0]
                    input_mapping = config_item[1] if len(config_item) > 1 else None
                    output_mapping = config_item[2] if len(config_item) > 2 else None
                else:
                    config = config_item
                    input_mapping = None
                    output_mapping = None
                
                # Determine node name
                node_name = self.node_names[i]
                
                # Determine next node
                if i < len(self.node_configs) - 1:
                    next_node = self.node_names[i+1]  # Connect to next node in sequence
                else:
                    next_node = END  # Last node goes to END
                
                # Create node function
                node_fn = create_node_function(
                    config=config,
                    input_mapping=input_mapping,
                    output_mapping=output_mapping,
                    next_node=next_node
                )
                
                # Add node to graph
                self.graph.add_node(node_name, node_fn)
                
                # Add edge to next node
                if i < len(self.node_configs) - 1:
                    self.graph.add_edge(node_name, self.node_names[i+1])
                else:
                    # Explicitly add edge to END
                    self.graph.add_edge(node_name, END)
            
        # Set entry point
        if isinstance(self.node_configs, dict):
            # Use first node from edge_map or first node in configs
            entry_candidates = [from_node for from_node, _ in self.edge_map]
            if entry_candidates:
                self.graph.set_entry_point(entry_candidates[0])
            else:
                self.graph.set_entry_point(next(iter(self.node_configs.keys())))
        else:
            # Use first node in the list
            self.graph.set_entry_point(self.node_names[0])
    
    def run(self, input_text: str):
        """
        Execute the agent with the provided input.
        Extends the BaseAgent run method to handle custom state initialization.
        """
        if not self.graph:
            raise RuntimeError("Workflow graph is not set up.")
        if not self.app:
            self.compile_workflow(checkpointer=self.memory)
        
        # Create initial state with input
        input_state = {"messages": [("user", input_text)]}
        
        # Add any custom state fields with defaults
        if isinstance(self.custom_state, dict):
            for key, value in self.custom_state.items():
                if key != "messages" and key not in input_state:
                    input_state[key] = value
        
        # Run the agent with the initialized state
        for output in self.app.stream(
            input_state,
            stream_mode="values",
            config=self.runnable_config,
            debug=True
        ):
            # Print messages as they come in
            if "messages" in output and output["messages"]:
                message = output["messages"][-1]
                if isinstance(message, tuple):
                    print(f"{message[0]}: {message[1]}")
                else:
                    message.pretty_print()
        
        # Auto-save state history after execution
        self.save_state_history()
        
        # Auto-save graph after execution
        self.visualize_graph()
        
        return output


# Register the agent
register_agent(SequentialAgentConfig, SequentialAgent)


# Factory functions for easier creation

def create_sequential_agent(
    configs: Union[List[Union[AugLLMConfig, Tuple[AugLLMConfig, Dict[str, str], Dict[str, str]]]], 
                Dict[str, Union[AugLLMConfig, Tuple[AugLLMConfig, Dict[str, str], Dict[str, str]]]]],
    node_names: Optional[List[str]] = None,
    edge_map: Optional[List[Tuple[str, Union[str, Literal["END"]]]]] = None,
    custom_state: Optional[Dict[str, Any]] = None
):
    """
    Create a sequential agent with the specified configuration.
    
    Args:
        configs: List or Dict of AugLLM configs or (config, input_mapping, output_mapping) tuples
        node_names: Optional node names to use (only if configs is a list)
        edge_map: Optional edge map for routing (required if configs is a dict)
        custom_state: Optional custom state fields
        
    Returns:
        SequentialAgent instance
    """
    # Process custom state
    #from state_schema_manager import StateSchemaManager
    if custom_state:
        schema_manager = StateSchemaManager(custom_state)
        state_schema = schema_manager.get_model()
    else:
        schema_manager = StateSchemaManager({"messages": []})
        state_schema = schema_manager.get_model()
    
    # Create configuration
    config = SequentialAgentConfig(
        engine=AugLLMConfig(llm_config=AzureLLMConfig(model="gpt-4o")),  # Default engine (not used)
        nodes=configs,
        node_names=node_names,
        edge_map=edge_map,
        custom_state=custom_state,
        state_schema=state_schema
    )
    
    return SequentialAgent(config)

/home/will/Projects/haive/backend/haive/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:345: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'populate_by_name'
* 'orm_mode' has been renamed to 'from_attributes'
  warnings.warn(message, UserWarning)


NameError: name 'BaseAgentConfig' is not defined